### Set catalog/schema context

In [0]:
spark.sql("USE CATALOG nyc_taxi_project")
spark.sql("USE SCHEMA raw")

DataFrame[]

### Load the zone lookup table first

In [0]:
spark.sql("USE CATALOG nyc_taxi_project")
spark.sql("USE SCHEMA raw")

zones = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/nyc_taxi_project/raw/taxi_landing/taxi_zone_lookup.csv")

zones.write.format("delta").mode("overwrite") \
  .saveAsTable("nyc_taxi_project.raw.dim_taxi_zones")

print("Write complete.")

Write complete.


In [0]:
spark.sql("SHOW TABLES IN nyc_taxi_project.raw").show(truncate=False)

+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|raw     |bronze_taxi_trips |false      |
|raw     |dim_taxi_zones    |false      |
|raw     |gold_borough_flow |false      |
|raw     |gold_daily_summary|false      |
|raw     |gold_hourly_demand|false      |
|raw     |gold_top_zones    |false      |
|raw     |silver_taxi_trips |false      |
+--------+------------------+-----------+



In [0]:
display(dbutils.fs.ls("/Volumes/nyc_taxi_project/raw/taxi_landing/"))

path,name,size,modificationTime
dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/taxi_zone_lookup.csv,taxi_zone_lookup.csv,12331,1783398270000
dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-01.parquet,yellow_tripdata_2024-01.parquet,49961641,1783398283000
dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-02.parquet,yellow_tripdata_2024-02.parquet,50349284,1783398283000
dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,yellow_tripdata_2024-03.parquet,60078280,1783440221000


In [0]:
spark.table("nyc_taxi_project.raw.dim_taxi_zones").show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


### Load the bronze table

In [0]:
bronze = spark.table("nyc_taxi_project.raw.bronze_taxi_trips")
print(f"Bronze row count: {bronze.count()}")

Bronze row count: 9554778


### Filter out bad/invalid rows

In [0]:
from pyspark.sql.functions import col

silver = (
    bronze
    .filter(col("passenger_count") > 0)          # no phantom trips with 0 passengers
    .filter(col("trip_distance") > 0)             # no zero-distance trips
    .filter(col("fare_amount") > 0)                # no negative/zero fares (refunds, bad meter data)
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))  # dropoff must be after pickup
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
)

print(f"Rows before cleaning: {bronze.count()}")
print(f"Rows after cleaning: {silver.count()}")

Rows before cleaning: 9554778
Rows after cleaning: 8479488


### Add derived columns (duration, speed, tip %)

In [0]:
from pyspark.sql.functions import unix_timestamp, round as spark_round

silver = (
    silver
    .withColumn("trip_duration_min",
        spark_round((unix_timestamp("tpep_dropoff_datetime") -
                      unix_timestamp("tpep_pickup_datetime")) / 60, 2))
    .withColumn("tip_pct",
        spark_round((col("tip_amount") / col("fare_amount")) * 100, 2))
)

# Now filter out trips where duration is nonsensical (e.g., 0 minutes or absurdly long)
silver = silver.filter(col("trip_duration_min") > 0).filter(col("trip_duration_min") < 240)  # cap at 4 hours

silver = silver.withColumn("avg_speed_mph",
    spark_round(col("trip_distance") / (col("trip_duration_min") / 60), 2))

### Join in zone names (pickup and dropoff)

In [0]:
from pyspark.sql.functions import col

# Re-read zones from the Delta table instead of relying on the in-memory variable
zones = spark.table("nyc_taxi_project.raw.dim_taxi_zones")

zones_pu = zones.select(
    col("LocationID").alias("PULocationID"),
    col("Borough").alias("pickup_borough"),
    col("Zone").alias("pickup_zone")
)

zones_do = zones.select(
    col("LocationID").alias("DOLocationID"),
    col("Borough").alias("dropoff_borough"),
    col("Zone").alias("dropoff_zone")
)

silver = silver.join(zones_pu, on="PULocationID", how="left") \
               .join(zones_do, on="DOLocationID", how="left")

In [0]:
print(silver.columns)

['DOLocationID', 'PULocationID', 'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'ingestion_time', 'source_file', 'trip_duration_min', 'tip_pct', 'avg_speed_mph', 'pickup_borough', 'pickup_zone', 'dropoff_borough', 'dropoff_zone']


### Select final clean columns

In [0]:
silver_final = silver.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID", "pickup_borough", "pickup_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone",
    "fare_amount",
    "tip_amount",
    "tip_pct",
    "total_amount",
    "trip_duration_min",
    "avg_speed_mph",
    "source_file",
    "ingestion_time"
)

### Data Quality Checks

In [0]:
row_count = silver_final.count()
assert row_count > 0, "Silver dataframe is empty — pipeline should stop here"

critical_nulls = silver_final.filter(
    col("fare_amount").isNull() |
    col("tpep_pickup_datetime").isNull() |
    col("PULocationID").isNull()
).count()
assert critical_nulls == 0, f"Found {critical_nulls} rows with nulls in critical columns"

bad_fares = silver_final.filter((col("fare_amount") > 500) | (col("fare_amount") < 0)).count()
print(f"Rows with suspicious fare amounts (>$500 or negative): {bad_fares}")

print(f"Data quality checks passed. {row_count} rows ready to write.")
# ═══════════════════════════════════════════

# ... then your EXISTING Step 5i write/MERGE cell continues after this ...
table_name = "nyc_taxi_project.raw.silver_taxi_trips"
if spark.catalog.tableExists(table_name):
    silver_final.createOrReplaceTempView("new_silver_data")
    ...

Rows with suspicious fare amounts (>$500 or negative): 68
Data quality checks passed. 8473894 rows ready to write.


### Write to the Silver Delta table

In [0]:
table_name = "nyc_taxi_project.raw.silver_taxi_trips"

try:
    if spark.catalog.tableExists(table_name):
        silver_final.createOrReplaceTempView("new_silver_data")
        spark.sql(f"""
            MERGE INTO {table_name} AS t
            USING new_silver_data AS s
            ON t.source_file = s.source_file
               AND t.VendorID = s.VendorID
               AND t.tpep_pickup_datetime = s.tpep_pickup_datetime
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"✅ Merge succeeded at {spark.sql('SELECT current_timestamp()').collect()[0][0]}")
    else:
        silver_final.write.format("delta").saveAsTable(table_name)
        print("✅ Table created for the first time.")
except Exception as e:
    print(f"❌ Silver pipeline failed: {str(e)}")
    raise

✅ Merge succeeded at 2026-07-09 03:23:58.250859


### Verify

In [0]:
result = spark.table("nyc_taxi_project.raw.silver_taxi_trips")
print(f"Total rows in silver table: {result.count()}")
display(result.limit(10))

# Quick sanity checks
result.select("pickup_borough").distinct().show()
result.groupBy("pickup_borough").count().orderBy(col("count").desc()).show()

Total rows in silver table: 8473894


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,pickup_borough,pickup_zone,DOLocationID,dropoff_borough,dropoff_zone,fare_amount,tip_amount,tip_pct,total_amount,trip_duration_min,avg_speed_mph,source_file,ingestion_time
2,2024-03-01T14:23:52.000,2024-03-01T14:50:02.000,2,1.55,163,Manhattan,Midtown North,100,Manhattan,Garment District,21.2,5.04,23.77,30.24,26.17,3.55,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
2,2024-03-01T14:23:52.000,2024-03-01T14:37:16.000,1,2.02,237,Manhattan,Upper East Side South,238,Manhattan,Upper West Side North,14.2,3.64,25.63,21.84,13.4,9.04,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
2,2024-03-01T22:16:33.000,2024-03-01T22:27:18.000,2,2.0,142,Manhattan,Lincoln Square East,236,Manhattan,Upper East Side North,12.8,3.56,27.81,21.36,10.75,11.16,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
2,2024-03-01T23:17:13.000,2024-03-01T23:40:57.000,2,3.35,144,Manhattan,Little Italy/NoLiTa,230,Manhattan,Times Sq/Theatre District,21.9,5.38,24.57,32.28,23.73,8.47,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
2,2024-03-01T23:17:13.000,2024-03-01T23:52:03.000,1,16.88,132,Queens,JFK Airport,170,Manhattan,Murray Hill,70.0,16.19,23.13,98.88,34.83,29.08,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
2,2024-03-01T23:17:13.000,2024-03-01T23:22:30.000,1,0.44,48,Manhattan,Clinton East,163,Manhattan,Midtown North,5.8,2.16,37.24,12.96,5.28,5.0,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
2,2024-03-01T23:17:13.000,2024-03-01T23:46:14.000,1,9.29,113,Manhattan,Greenwich Village North,89,Brooklyn,Flatbush/Ditmas Park,41.5,0.0,0.0,53.44,29.02,19.21,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
1,2024-03-02T01:57:33.000,2024-03-02T02:18:44.000,1,4.0,113,Manhattan,Greenwich Village North,49,Brooklyn,Clinton Hill,21.2,5.25,24.76,31.45,21.18,11.33,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
1,2024-03-02T11:24:14.000,2024-03-02T11:42:55.000,1,3.7,262,Manhattan,Yorkville East,186,Manhattan,Penn Station/Madison Sq West,21.2,3.0,14.15,28.2,18.68,11.88,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z
1,2024-03-02T12:41:18.000,2024-03-02T12:51:34.000,1,1.2,263,Manhattan,Yorkville West,140,Manhattan,Lenox Hill East,10.7,2.9,27.1,17.6,10.27,7.01,dbfs:/Volumes/nyc_taxi_project/raw/taxi_landing/yellow_tripdata_2024-03.parquet,2026-07-07T16:15:05.170Z


+--------------+
|pickup_borough|
+--------------+
|           N/A|
|        Queens|
|         Bronx|
|           EWR|
|       Unknown|
|     Manhattan|
| Staten Island|
|      Brooklyn|
+--------------+

+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|7605205|
|        Queens| 766069|
|      Brooklyn|  58035|
|       Unknown|  27321|
|         Bronx|  15505|
|           N/A|   1478|
|           EWR|    179|
| Staten Island|    102|
+--------------+-------+



# Checklist before Gold

In [0]:
spark.sql("USE CATALOG nyc_taxi_project")
spark.sql("USE SCHEMA raw")
spark.sql("SHOW TABLES IN nyc_taxi_project.raw").show(truncate=False)

+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|raw     |bronze_taxi_trips |false      |
|raw     |dim_taxi_zones    |false      |
|raw     |gold_borough_flow |false      |
|raw     |gold_daily_summary|false      |
|raw     |gold_hourly_demand|false      |
|raw     |gold_top_zones    |false      |
|raw     |silver_taxi_trips |false      |
|        |new_silver_data   |true       |
+--------+------------------+-----------+



In [0]:
silver_check = spark.table("nyc_taxi_project.raw.silver_taxi_trips")

print(f"Total rows: {silver_check.count()}")
silver_check.printSchema()

Total rows: 8473894
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tip_pct: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import col

total = silver_check.count()
null_pickup = silver_check.filter(col("pickup_borough").isNull()).count()
null_dropoff = silver_check.filter(col("dropoff_borough").isNull()).count()

print(f"Total: {total}")
print(f"Missing pickup_borough: {null_pickup} ({round(null_pickup/total*100,2)}%)")
print(f"Missing dropoff_borough: {null_dropoff} ({round(null_dropoff/total*100,2)}%)")

Total: 8473894
Missing pickup_borough: 0 (0.0%)
Missing dropoff_borough: 0 (0.0%)


In [0]:
display(silver_check.select(
    "PULocationID", "pickup_borough", "pickup_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone",
    "fare_amount", "trip_duration_min", "tip_pct"
).limit(15))

PULocationID,pickup_borough,pickup_zone,DOLocationID,dropoff_borough,dropoff_zone,fare_amount,trip_duration_min,tip_pct
163,Manhattan,Midtown North,100,Manhattan,Garment District,21.2,26.17,23.77
237,Manhattan,Upper East Side South,238,Manhattan,Upper West Side North,14.2,13.4,25.63
142,Manhattan,Lincoln Square East,236,Manhattan,Upper East Side North,12.8,10.75,27.81
144,Manhattan,Little Italy/NoLiTa,230,Manhattan,Times Sq/Theatre District,21.9,23.73,24.57
132,Queens,JFK Airport,170,Manhattan,Murray Hill,70.0,34.83,23.13
48,Manhattan,Clinton East,163,Manhattan,Midtown North,5.8,5.28,37.24
113,Manhattan,Greenwich Village North,89,Brooklyn,Flatbush/Ditmas Park,41.5,29.02,0.0
113,Manhattan,Greenwich Village North,49,Brooklyn,Clinton Hill,21.2,21.18,24.76
262,Manhattan,Yorkville East,186,Manhattan,Penn Station/Madison Sq West,21.2,18.68,14.15
263,Manhattan,Yorkville West,140,Manhattan,Lenox Hill East,10.7,10.27,27.1


### rows dropped during cleaning

In [0]:
bronze_count = spark.table("nyc_taxi_project.raw.bronze_taxi_trips").count()
silver_count = silver_check.count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Rows filtered out during cleaning: {bronze_count - silver_count} ({round((bronze_count-silver_count)/bronze_count*100,2)}%)")

Bronze rows: 9554778
Silver rows: 8473894
Rows filtered out during cleaning: 1080884 (11.31%)
